# P12 ユースケース 5 本の評価カード

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

各ラボの本物の実行の結果（out/）から、5 つの業務（すべて架空の会社）の評価カードを作り、Sonnet と Opus を全ラボで並べます。LLM は呼びません。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 評価カードの書き方
`build_cards.py` を回して、カードの欄と UC1 の中身を見ます。

In [ ]:
def cut(s, w=100):                  # 表示幅（全角 2）で切って、後ろを「…」にする
    s, n = s.replace("\n", " "), 0
    for i, ch in enumerate(s):
        n += 2 if ord(ch) > 0x2E80 else 1
        if n > w:
            return s[:i] + "…"
    return s
import io, contextlib
from common import kit
bc = lab("build_cards", "lab12_usecases")
with contextlib.redirect_stdout(io.StringIO()):
    md = bc.main([])
cards = md.split("\n## ")[1:]
print("カード:", " / ".join(c.splitlines()[0].split("（")[0] for c in cards))
print("\n■", cards[0].splitlines()[0])
for line in cards[0].splitlines():
    if line.startswith("- **") and "実測" not in line:
        print(cut("  " + line[2:].replace("**", ""), 100))

## 2. 問い合わせ分類・請求書・規程の質問のカード（実測の欄）
UC1〜UC3。数字は各ラボの out/ から。

In [ ]:
def measured(c):
    lines = c.splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith("- **実測**"))
    return [l.strip()[2:] for l in lines[i + 1:] if l.startswith("  - ")]

for c in cards[:3]:
    print("■", c.splitlines()[0])
    for m in measured(c):
        print("   " + cut(m, 96))

## 3. 発注エージェント・返信の下書きのカード（実測の欄）
UC4・UC5。模擬の判定者は人ではありません。

In [ ]:
for c in cards[3:]:
    print("■", c.splitlines()[0])
    for m in measured(c):
        print("   " + cut(m, 96))

## 4. Sonnet か Opus か ― 全ラボの比較表（その 1）
分類・抽出・RAG・エージェント。

In [ ]:
J = lambda p: kit.read_json(p)
L3, L4, L5 = "lab03_classify_extract/out/", "lab04_rag_eval/out/", "lab05_agent_trajectory/out/"
def two(f):
    return [f("sonnet-5-5"), f("opus-5-5")]
rows = [
    ["分類 v1 120 件", *two(lambda m: f"{J(L3 + f'summary_classify_{m}.json')['accuracy']:.0%}・{J(L3 + f'summary_classify_{m}.json')['cost_usd']:.2f} USD")],
    ["分類 v2 難問 48 件", *two(lambda m: f"{J(L3 + f'summary_classify_v2_{m}.json')['accuracy']:.0%}・{J(L3 + f'summary_classify_v2_{m}.json')['cost_usd']:.2f} USD")],
    ["抽出 48 件（全項目）", *two(lambda m: f"{J(L3 + f'summary_extract_{m}.json')['strict']['doc_all_correct']:.0%}・{J(L3 + f'summary_extract_{m}.json')['cost_usd']:.2f} USD")],
    ["RAG 30 問（照合 v1 → v2）", *two(lambda m: f"{J(L4 + f'summary_rag_normal_{m}.json')['correct']} → {J(L4 + f'summary_rag_normal_{m}.json')['correct_v2']}")],
    ["根拠を抜いた 26 問で断った", *two(lambda m: f"{J(L4 + f'summary_rag_degraded_{m}.json')['abstained']}/26")],
    ["発注 10 件×3 回", *two(lambda m: f"{J(L5 + f'summary_agent_{m}_k3.json')['pass']}/30・{J(L5 + f'summary_agent_{m}_k3.json')['cost_usd']:.2f} USD")],
]
print(kit.table(rows, ["実験", "Sonnet 5.5", "Opus 5.5"]))

## 5. Sonnet か Opus か ― 全ラボの比較表（その 2）
判定役・依頼文が雑になったとき・安全。

In [ ]:
L6, L7, L9, L11 = "lab06_judge_bias/out/", "lab07_judge_calibration/out/", "lab09_ci_gate/out/", "lab11_safety/out/"
rows = [
    ["判定役・素朴な依頼文で一貫", *two(lambda m: f"{J(L6 + f'summary_bias_naive_{m}.json')['result']['equal']['consistent']}/20")],
    ["判定 v1・dev の真陰性", *two(lambda m: f"{J(L7 + f'summary_calib_v1_dev_{m}.json')['tn']}/22")],
    ["短い依頼文の候補・v2 48 問", *two(lambda m: f"{sum(r['correct'] for r in kit.read_jsonl(L9 + f'runs_candidate_v2_{m}.jsonl'))}/48")],
    ["漏えい / 止めすぎ（各 12 件）", *two(lambda m: f"{J(L11 + f'summary_safety_{m}.json')['leaks']}/12 ・ {J(L11 + f'summary_safety_{m}.json')['over_refusals']}/12")],
]
print(kit.table(rows, ["実験", "Sonnet 5.5", "Opus 5.5"]))
print("\n手引きつきの試験は両モデルとも飽和。差が出たのは費用・待ち時間・雑な依頼文での崩れ方・判定役の揺れ")